# 5G Network Intelligence — Cell Spatial–Temporal Attention

## Hypothesis
Raw histories from neighbouring beams in the same cell may carry predictive information that isolated per-beam models miss. Test a shared patch temporal encoder with within-cell attention against the identical model with off-diagonal attention blocked. Improvement is a hypothesis, not a promise.


## Frozen temporal protocol
Fit/stop/score cutoffs: 335/383/443 and 395/443/503. Context 168 hours, four KPIs and missing masks, horizons 1/6/24h. All 2,880 beams, grouped by known station/cell IDs into 90 cells of 32 beams. No future actual inputs, fitted per-beam statistics or identity embeddings. Original values from hour 504 onward are redacted in cloud inputs.


In [ ]:
# Safe review: Run All does not retrain.
RUN_TRAINING = False


## Configuration and compute budget
Two matched modes across two folds: four hosted GPU fits. Maximum 100 epochs, stopping patience 12, seed 17. No budget extension or architecture selection based on scoring results. Choose best weights using stopping MAE only.


In [ ]:
from pathlib import Path
import json
import yaml
import pandas as pd
from IPython.display import display
PROJECT = Path.cwd()
if (PROJECT / 'configs/cell_attention.yaml').exists():
    protocol = yaml.safe_load((PROJECT / 'configs/cell_attention.yaml').read_text(encoding='utf-8'))
    display(protocol)


## Causal preprocessing
Fit one global median/mean/std per KPI on each fit prefix. Preserve observed zeros; add missing masks. A training sample contains only t−167 through t for all beams of one cell. Future labels are used only for supervised loss. Cell membership is topology, not a fitted numerical beam summary. Phase4 supplies known daily/weekly calendar positions.


## Model and objective
Shared 12-hour patches with stride 6, width 64, two temporal transformer layers, and four-head cell attention. Both modes have identical parameter counts and initialization. The local control blocks access to other beams. A causal seasonal baseline plus residual head produces nonnegative DLPRB predictions. Optimize masked MAE directly in original DLPRB units. Implementation: `src/models/cell_attention.py`; inputs: `src/features/cell_inputs.py`.


## Training and debugging
Hosted Colab/Kaggle only. Save best/last weights, optimizer and RNG each epoch to durable storage. Logs include fold/mode, parameter count, cell-window population, losses and epoch duration. Historical trainer log names ending in `scaled` represent physical DLPRB MAE in this study. An epoch-capped run is not evidence of completed early stopping.


In [ ]:
if RUN_TRAINING:
    import subprocess, sys
    (PROJECT / 'reports').mkdir(exist_ok=True)
    with (PROJECT / 'reports/cell_attention_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.train_cell_attention'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        TRAIN_EXIT = process.wait()
else:
    print('Review mode: training skipped.')


## Evaluation
Full frozen scoring origins and all original beams. MAE: original DLPRB units. WAPE: sum absolute error / sum absolute actual. Report all horizons, zero false positives and observed-target coverage. Independent forecast reconstruction and CPU/GPU parity are required before acceptance. Compare with the fixed current MLP-B/hurdle blend and matched local mode.


In [ ]:
path = PROJECT / 'reports/cell_attention_development.csv'
if path.exists():
    results = pd.read_csv(path)
    results['WAPE (%)'] = 100 * results['wape']
    display(results)
else:
    print('No completed scoring results available.')


## Decision and limitations
Seek at least 5% mean MAE reduction with no horizon regression; the initial 2% research gate only qualifies a candidate for further investigation. Require additional seeds and paired uncertainty checks before superiority claims. Reused development data is not fresh confirmation. Preserve negative results; no automatic promotion, final validation refit or holdout evaluation. More training does not guarantee improvement.
